In [29]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
import polars as pl
from pathlib import Path
from datetime import datetime,timedelta
from vnpy.alpha import Segment, AlphaDataset
import pandas as pd
import numpy as np
import lightgbm as lgb
from factor_define import (
    FACTOR_REGISTRY1,
    FACTOR_NAMES1,
    FACTOR_NAMES2
)
import pickle
import gc

In [30]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [31]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime.now()
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [32]:
# ============================================================================
# Cell 4: 加载数据集
# ============================================================================
DATASET_NAME = 'v5'
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [33]:
print(dataset.learn_df)

shape: (601_000, 31)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ avg_trade ┆ big_order ┆ intraday_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ _size     ┆ _net      ┆ price_eff ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iciency   ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.196827  ┆ -0.299025 ┆ … ┆ 2.476964  ┆ 0.078719  ┆ 0.412051  ┆ -2.06371 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 4        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆     

In [34]:
print(dataset.result_df)

shape: (1_051_119, 32)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ open      ┆ late_skew ┆ … ┆ avg_trade ┆ big_order ┆ intraday_ ┆ label    │
│ ---       ┆ ---       ┆ ---       ┆ _ret      ┆   ┆ _size     ┆ _net      ┆ price_eff ┆ ---      │
│ datetime[ ┆ str       ┆ f64       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iciency   ┆ f64      │
│ μs]       ┆           ┆           ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 10.551859 ┆ 0.16691   ┆ … ┆ 1.1906e7  ┆ 0.156992  ┆ 0.091623  ┆ -0.02986 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 2        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆   

In [35]:
print(dataset.infer_df)

shape: (604_800, 31)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ avg_trade ┆ big_order ┆ intraday_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ _size     ┆ _net      ┆ price_eff ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iciency   ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.206012  ┆ -0.299025 ┆ … ┆ 2.527748  ┆ 0.087895  ┆ 0.453815  ┆ -2.03216 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 6        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆     

In [36]:
# ============================================================================
# Cell 4 修改版：提取 LambdaRank 所需的数据（按天等频分档）
# ============================================================================
def extract_lambdarank_data(dataset, segment, n_quantiles=5):
    """
    从 AlphaDataset 提取 LambdaRank 所需数据（稳健版）
    使用 Polars 的 rank + 线性映射，避免 qcut 问题
    """
    df = dataset.fetch_learn(segment)
    df = df.sort('datetime')

    # 方案：每天内按收益率排序，然后等频分成 n_quantiles 档
    # 使用 rank('ordinal') 得到每个样本在当天的唯一排名（从1开始）
    df = df.with_columns(
        pl.col('label').rank('ordinal').over('datetime').alias('_rank')
    )
    # 计算每天的总样本数
    df = df.with_columns(
        pl.col('label').count().over('datetime').alias('_day_count')
    )
    # 将排名映射到 0 ~ n_quantiles-1
    # 公式：floor( (rank - 1) / (day_count - 1) * (n_quantiles - 1) )
    # 注意：当 day_count == 1 时，分母为0，需要特殊处理
    df = df.with_columns(
        pl.when(pl.col('_day_count') == 1)
        .then(n_quantiles // 2)   # 只有一只股票时给中间档位
        .otherwise(
            ((pl.col('_rank') - 1) / (pl.col('_day_count') - 1) * (n_quantiles - 1))
            .cast(pl.Int64)
        )
        .alias('rank_label')
    )

    # 验证标签是否在 0 ~ n_quantiles-1 范围内
    print(f"{segment.name}: 标签唯一值 = {df['rank_label'].unique().to_numpy()}")

    # 提取特征和元数据
    meta_cols = ['datetime', 'vt_symbol']
    df_meta = df.select(meta_cols)
    exclude_cols = ['datetime', 'vt_symbol', 'label', 'rank_label', '_rank', '_day_count']
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    X = df.select(feature_cols).to_numpy()
    y = df['rank_label'].to_numpy()
    date_codes = df['datetime'].to_numpy()
    unique_dates, group_sizes = np.unique(date_codes, return_counts=True)

    print(f'{segment.name}: X.shape={X.shape}, y 取值 {np.unique(y)}')
    print(f'交易日数量 = {len(unique_dates)}, 平均每天样本数 = {group_sizes.mean():.1f}')
    return X, y, df_meta, group_sizes

# 提取数据（注意变量名保持一致）
n_quantiles = 30
print('提取 LambdaRank 训练数据...')
X_train, y_train, meta_train, group_train = extract_lambdarank_data(dataset, Segment.TRAIN, n_quantiles=n_quantiles)

print('\n提取验证数据...')
X_valid, y_valid, meta_valid, group_valid = extract_lambdarank_data(dataset, Segment.VALID, n_quantiles=n_quantiles)

print('\n提取测试数据...')
X_test, y_test, meta_test, group_test = extract_lambdarank_data(dataset, Segment.TEST, n_quantiles=n_quantiles)



提取 LambdaRank 训练数据...
TRAIN: 标签唯一值 = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
TRAIN: X.shape=(433677, 28), y 取值 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
交易日数量 = 1457, 平均每天样本数 = 297.7

提取验证数据...
VALID: 标签唯一值 = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
VALID: X.shape=(72461, 28), y 取值 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
交易日数量 = 242, 平均每天样本数 = 299.4

提取测试数据...
TEST: 标签唯一值 = [nan  0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16.
 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29.]
TEST: X.shape=(94862, 28), y 取值 [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. nan]
交易日数量 = 317, 平均每天样本数 = 299.2


In [37]:
# ============================================================================
# Cell 5: 训练 LambdaRank 模型
# ============================================================================
print('\n开始训练 LambdaRank 模型...')

# 创建 Dataset，直接传入 group 参数
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=group_valid, reference=train_data)

# LambdaRank 专用参数（重点：objective, metric, label_gain）
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',                 # 评估指标
    'ndcg_eval_at': [1, 3, 5, 7, 10],        # 计算 NDCG@1, @3, @5
    'label_gain': [2**i-1 for i in range(n_quantiles)],   # 对应 5 档标签的增益 (2^rel - 1)
    'lambdarank_truncation_level': n_quantiles, # 截断级别，与 label_gain 长度一致

    #
    'num_leaves': 1024,
    'max_depth': -1,
    'min_data_in_leaf': 300,

    # 学习参数
    'learning_rate': 0.001,
    'feature_fraction': 0.88,
    'bagging_fraction': 0.87,
    'bagging_freq': 5,

    # 正则化
    'lambda_l1': 30,
    'lambda_l2': 0.0,

    # 其他
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbose': -1,
    'seed': 42,
    'num_threads': -1
}

num_boost_round = 1000
early_stopping_rounds = 100

# 训练（这里仍然可以用 feval 监控 IC，但不是必须）
model = lgb.train(
    params,
    train_data,
    num_boost_round=num_boost_round,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(early_stopping_rounds),
        lgb.log_evaluation(period=1)
    ]
)

print(f'\n训练完成！最佳迭代轮数: {model.best_iteration}')
if model.best_score:
    print(f"最佳验证 NDCG: {model.best_score['valid']}")


开始训练 LambdaRank 模型...
[1]	train's ndcg@1: 0.0424556	train's ndcg@3: 0.0617477	train's ndcg@5: 0.0662378	train's ndcg@7: 0.0709017	train's ndcg@10: 0.0744764	valid's ndcg@1: 0.0469642	valid's ndcg@3: 0.0657935	valid's ndcg@5: 0.0692239	valid's ndcg@7: 0.0693723	valid's ndcg@10: 0.0707171
Training until validation scores don't improve for 100 rounds
[2]	train's ndcg@1: 0.0632166	train's ndcg@3: 0.0855042	train's ndcg@5: 0.094455	train's ndcg@7: 0.0993548	train's ndcg@10: 0.103696	valid's ndcg@1: 0.0584251	valid's ndcg@3: 0.0824847	valid's ndcg@5: 0.0921265	valid's ndcg@7: 0.0982008	valid's ndcg@10: 0.101507
[3]	train's ndcg@1: 0.0896117	train's ndcg@3: 0.118672	train's ndcg@5: 0.127137	train's ndcg@7: 0.129459	train's ndcg@10: 0.130132	valid's ndcg@1: 0.0707955	valid's ndcg@3: 0.105223	valid's ndcg@5: 0.110934	valid's ndcg@7: 0.112184	valid's ndcg@10: 0.113859
[4]	train's ndcg@1: 0.0950479	train's ndcg@3: 0.129706	train's ndcg@5: 0.135662	train's ndcg@7: 0.138391	train's ndcg@10: 0.1387

In [38]:
# ============================================================================
# Cell 6: 特征重要性分析
# ============================================================================

print('\n特征重要性分析...')
lag_days = 0
# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = [f'{factor}_lag_{lag}' for factor in FACTOR_NAMES1 for lag in range(0, lag_days + 1)]

# 创建重要性 DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print('Top 20 重要特征:')
print(importance_df.head(120))


特征重要性分析...
Top 20 重要特征:
                            feature    importance
25             avg_trade_size_lag_0  28404.388617
3         corr_close_nextopen_lag_0   4521.480298
13         mmt_top20VolumeRet_lag_0   3721.082995
1               down_vol_perc_lag_0   2200.864186
14               liq_closevol_lag_0   1229.981982
22            trade_CBuyRatio_lag_0    930.525302
21                  crowd_fft_lag_0    780.801673
24        trade_top20retRatio_lag_0    727.007082
11      corr_volume_amplitude_lag_0    714.147814
23          trade_netBuyRatio_lag_0    530.938877
2            corr_ret_lastret_lag_0    415.992387
16      price_range_vol_ratio_lag_0    209.737903
19                   corr_prv_lag_0    147.069209
15                   vol_skew_lag_0    132.520448
6                volume_perc4_lag_0    108.066707
7                volume_perc5_lag_0    104.075391
8                volume_perc6_lag_0     95.571555
18                   corr_pvl_lag_0     79.412381
17                    cor

In [39]:
# ============================================================================
# Cell 7: 生成回测信号
# ============================================================================
print('\n在测试集上预测...')

# 预测
predictions = model.predict(X_test, num_iteration=model.best_iteration)

print(f'预测完成，预测样本数: {len(predictions)}')

# 构建信号 DataFrame
signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])

print(f'\n信号数据形状: {signal.shape}')
print('信号数据预览:')
print(signal.head(10))


在测试集上预测...
预测完成，预测样本数: 94862

信号数据形状: (94862, 3)
信号数据预览:
shape: (10, 3)
┌─────────────────────┬─────────────┬───────────┐
│ datetime            ┆ vt_symbol   ┆ signal    │
│ ---                 ┆ ---         ┆ ---       │
│ datetime[μs]        ┆ str         ┆ f64       │
╞═════════════════════╪═════════════╪═══════════╡
│ 2025-01-02 00:00:00 ┆ 000001.SZSE ┆ -0.014184 │
│ 2025-01-02 00:00:00 ┆ 000002.SZSE ┆ -0.027339 │
│ 2025-01-02 00:00:00 ┆ 000063.SZSE ┆ 0.010588  │
│ 2025-01-02 00:00:00 ┆ 000100.SZSE ┆ -0.002241 │
│ 2025-01-02 00:00:00 ┆ 000157.SZSE ┆ -0.027339 │
│ 2025-01-02 00:00:00 ┆ 000166.SZSE ┆ -0.025508 │
│ 2025-01-02 00:00:00 ┆ 000301.SZSE ┆ -0.013873 │
│ 2025-01-02 00:00:00 ┆ 000333.SZSE ┆ 0.009512  │
│ 2025-01-02 00:00:00 ┆ 000338.SZSE ┆ -0.014573 │
│ 2025-01-02 00:00:00 ┆ 000408.SZSE ┆ -0.013255 │
└─────────────────────┴─────────────┴───────────┘


In [40]:
# ============================================================================
# Cell 8: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v7'
SIGNAL_NAME = 'v7'

lab.save_model(MODEL_NAME, model)
lab.save_signal(SIGNAL_NAME, signal)
# # 保存 LightGBM 模型
# MODEL_PICKLE_PATH = LAB_PATH / 'model' / f'{MODEL_NAME}.pkl'
# with open(MODEL_PICKLE_PATH, 'wb') as f:
#     pickle.dump({
#         'model': model,
#         'params': params,
#         'best_iteration': model.best_iteration,
#         'best_score': model.best_score
#     }, f)
# print(f'模型已保存: {MODEL_PICKLE_PATH}')
#
# # 保存信号
# SIGNAL_PARQUET_PATH = LAB_PATH / 'signal' / f'{SIGNAL_NAME}.parquet'
# SIGNAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
# signal.write_parquet(str(SIGNAL_PARQUET_PATH))
# print(f'信号已保存: {SIGNAL_PARQUET_PATH}')

In [41]:
with pd.option_context('display.max_rows', None):
    print(importance_df.head(120))

                            feature    importance
25             avg_trade_size_lag_0  28404.388617
3         corr_close_nextopen_lag_0   4521.480298
13         mmt_top20VolumeRet_lag_0   3721.082995
1               down_vol_perc_lag_0   2200.864186
14               liq_closevol_lag_0   1229.981982
22            trade_CBuyRatio_lag_0    930.525302
21                  crowd_fft_lag_0    780.801673
24        trade_top20retRatio_lag_0    727.007082
11      corr_volume_amplitude_lag_0    714.147814
23          trade_netBuyRatio_lag_0    530.938877
2            corr_ret_lastret_lag_0    415.992387
16      price_range_vol_ratio_lag_0    209.737903
19                   corr_prv_lag_0    147.069209
15                   vol_skew_lag_0    132.520448
6                volume_perc4_lag_0    108.066707
7                volume_perc5_lag_0    104.075391
8                volume_perc6_lag_0     95.571555
18                   corr_pvl_lag_0     79.412381
17                    corr_pv_lag_0     48.577608
